In [1]:
import numpy as np
from scipy.integrate import fixed_quad, quad
import os
import plotly.graph_objects as go
from scipy.special import j0, jv
import pandas as pd

In [2]:
# Leitura do arquivo com separação por espaços
data_atlas = pd.read_csv(
    "../../data/sigma_tot_2/ensemble_StRh_atlas.dat",
    delim_whitespace=True,
    header=None,
    nrows=70  # lê apenas as 70 primeiras linhas
)

x_atlas = data_atlas[0].to_numpy()
y_atlas = data_atlas[1].to_numpy()
y_error_atlas = data_atlas[2].to_numpy()

In [3]:
# === Global Configuration and Constants ===
start_sqrt_s = 101  # Global parameter controlling energy scale
b_0 = (33 - 6) / (12 * np.pi)  # β0 for nf=3
Lambda = 0.284  # ΛQCD in GeV
gamma_1 = 0.084
gamma_2 = 2.36
rho = 4.0

lst_sigma_tot_born = []
lst_sqrt_s = []
lst_error = []

s0 = 1.0  # GeV^2

epsilon_atlas = 0.0732

model_params = {
    'atlas': {
        'pl':  {'mg': 0.417, 'a1': 1.563, 'a2': 2.22}
    }
}

epsilon_values = {
    'atlas': epsilon_atlas
}


In [4]:

# === Auxiliary Functions for Physical Model ===
def m2_pl(q2, mg):
    lambda_squared = Lambda ** 2
    rho_mg_squared = rho * mg ** 2
    ratio = np.log((q2 + rho_mg_squared) / lambda_squared) / np.log(rho_mg_squared / lambda_squared)
    return (mg ** 4 / (q2 + mg ** 2)) * ratio ** (gamma_2 - 1)

def get_m2_function(mass_model):
    return m2_pl

def G_p(q2, a1, a2):
    return np.exp(-(a1 * q2 + a2 * q2 ** 2))

def alpha_D(q2, mg, m2_func):
    m2 = m2_func(q2, mg)
    return 1.0 / (b_0 * (q2 + m2) * np.log((q2 + 4 * m2) / (Lambda ** 2)))

def T_1(k, q, phi, mg, a1, a2, m2_func):
    q2 = q ** 2
    qk_cos = q * k * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2

    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)
    G0 = G_p(q2, a1, a2)

    return alpha_D_plus * alpha_D_minus * G0 ** 2

def T_2(k, q, phi, mg, a1, a2, m2_func):
    q2 = q ** 2
    qk_cos = q * k * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2

    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)

    factor = q2 + 9 * abs(k ** 2 - q2 / 4)

    G0 = G_p(q2, a1, a2)
    G_minus = G_p(factor, a1, a2)

    return alpha_D_plus * alpha_D_minus * G_minus * (2 * G0 - G_minus)

def integrand(y, x, mg, a1, a2, m2_func):
    k = sqrt_s * x
    phi = 2 * np.pi * y
    jacobian = 2 * np.pi * sqrt_s

    return k * (T_1(k, 0.0, phi, mg, a1, a2, m2_func) - T_2(k, 0.0, phi, mg, a1, a2, m2_func)) * jacobian

def amp_calculation(diff_T, s, epsilon):
    alpha_pomeron = 1.0 + epsilon
    regge_factor = (s / s0) ** alpha_pomeron
    
    return 1j * 8.0 * regge_factor * diff_T

def sigma_tot(amp_value, s):
    return amp_value.imag / s * 0.389379323


In [5]:
lst_amp_born = []

lst_sqrt_s = []

lst_sigma_tot_born = []

# === Main Function ===
def main():
    global start_sqrt_s
    global sqrt_s

    max_sqrt_s = 13000
    step = 100
    n_points = 10000

    # Using only PL model with ATLAS
    mass_model = 'pl'
    ensemble = 'atlas'

    fig = go.Figure()

    m2_func = get_m2_function(mass_model)
    params = model_params[ensemble][mass_model]
    mg, a1, a2 = params['mg'], params['a1'], params['a2']
    epsilon = epsilon_values[ensemble]

    sqrt_s = start_sqrt_s
    while sqrt_s <= max_sqrt_s:
        def inner_integral(x):
            return fixed_quad(
                lambda y: integrand(y, x, mg, a1, a2, m2_func),
                0, 1,
                n=n_points
            )[0]

        integral_value = fixed_quad(
            inner_integral,
            0, 1,
            n=n_points
        )[0]

        diff_T = integral_value
        s = sqrt_s * sqrt_s

        amp_value = amp_calculation(diff_T, s, epsilon)
        sigma_tot_value = sigma_tot(amp_value, s)

        lst_sigma_tot_born.append(sigma_tot_value)
        lst_sqrt_s.append(sqrt_s)
        lst_amp_born.append(amp_value)

        sqrt_s += step

    # Add PL model trace
    fig.add_trace(go.Scatter(
        x=lst_sqrt_s,
        y=lst_sigma_tot_born,
        mode='lines+markers',
        line=dict(
            color='blue',
            width=2
        ),
        marker=dict(
            size=4
        ),
        name='PL Model (ATLAS)'
    ))

    # Add ATLAS data
    fig.add_trace(go.Scatter(
        x=x_atlas,
        y=y_atlas,
        mode='markers',
        marker=dict(
            color='black',
            size=6,
            symbol='square'
        ),
        error_y=dict(
            type='data',
            array=y_error_atlas,
            visible=True
        ),
        name='ATLAS Data'
    ))

    # Configure layout
    fig.update_layout(
        title='Sigma Tot vs. sqrt(s) - PL Model with ATLAS Data',
        xaxis=dict(
            title='sqrt(s) [GeV]',
            type='log',
        ),
        yaxis=dict(
            title='Sigma Tot [mb]',
        ),
        showlegend=True,
        legend=dict(
            title='Model/Data'
        ),
        plot_bgcolor='white',
        hovermode='x unified'
    )
    
    fig.update_xaxes(gridcolor='lightgray')
    fig.update_yaxes(gridcolor='lightgray')

    # fig.show(renderer="browser")
    # fig.write_html("results/sigma_tot/sigma_tot_pl_atlas.html")
    # fig.write_image("results/sigma_tot/sigma_tot_pl_atlas.pdf", width=1200, height=600)


In [6]:
if __name__ == "__main__":
    main()

In [7]:
def add_iterative_curve(fig, x_data, y_data, title:str, x_axis_name:str, y_axis_name:str, curve_name:str):

    fig.add_trace(go.Scatter(
    x = x_data,
    y = y_data,
    mode='lines+markers', 
    name=curve_name,
    marker=dict(size=4))
)

    fig.update_layout(
        title=title,
        xaxis=dict(
            title = x_axis_name,
            type='log',
        ),
        yaxis=dict(
            title= y_axis_name,
            # range=[80, 120]
        ),
        showlegend=True,
        plot_bgcolor='white',
        hovermode='x unified'
    )
        
    fig.update_xaxes(gridcolor='lightgray')
    fig.update_yaxes(gridcolor='lightgray')



In [8]:
def sigma_tot_eik(s, amp):
    return (4*np.pi)/s * amp.imag * 0.389379323

lst_s = [val**2 for val in lst_sqrt_s]


In [9]:
import numpy as np
from scipy.integrate import quad
from scipy.special import j0

# -------------------------------
# Parâmetros globais de integração
# -------------------------------
q_upper = 0.1
b_upper = 8

epsabs = 1e-10
epsrel = 1e-10
limit = 100_000

# -------------------------------
# Integral interna (sobre q)
# -------------------------------
def inner_integral(b, s, amp, epsabs=epsabs, epsrel=epsrel, limit=limit):
    integrand = lambda q: q * j0(b*q)
    result, _ = quad(integrand, 0, q_upper, epsabs=epsabs, epsrel=epsrel, limit=limit)
    return result * (1/s) * amp

# -------------------------------
# Integral externa (sobre b)
# -------------------------------
def outer_integrand(b, s, amp, epsabs=epsabs, epsrel=epsrel, limit=limit):
    chi = inner_integral(b, s, amp, epsabs=epsabs, epsrel=epsrel, limit=limit)
    return 1j * s * b * (1 - np.exp(1j * chi))

def outer_real(b, s, amp, **kwargs):
    return np.real(outer_integrand(b, s, amp, **kwargs))

def outer_imag(b, s, amp, **kwargs):
    return np.imag(outer_integrand(b, s, amp, **kwargs))

# -------------------------------
# Integral dupla
# -------------------------------
def compute_double_integral(s, amp, epsabs=epsabs, epsrel=epsrel, limit=limit):
    real_part, _ = quad(outer_real, 0, b_upper, args=(s, amp), epsabs=epsabs, epsrel=epsrel, limit=limit)
    imag_part, _ = quad(outer_imag, 0, b_upper, args=(s, amp), epsabs=epsabs, epsrel=epsrel, limit=limit)
    return real_part + 1j * imag_part



In [10]:
lst_amp_eik = [compute_double_integral(s, amp) for s, amp in zip(lst_s, lst_amp_born)]

lst_sigma_tot_eik = [sigma_tot_eik(s_val, amp_eik_val) for amp_eik_val, s_val in zip(lst_amp_eik, lst_s)]

lst_amp_eik_im = [amp.imag for amp in lst_amp_eik]

In [11]:
# fig_amp_eik = go.Figure()

# add_iterative_curve(fig_amp_eik, lst_sqrt_s, lst_amp_eik_im, 'Amp eikonal vs sqrt(s)', 'sqrt s [GeV]', 'Im(Amp_eikonal)', 'Amp Eikonal')

# # fig_amp_eik.show(renderer = 'browser')

# # fig.write_image(save_path, width=1200, height=600)
# fig_amp_eik.write_html("amp_eikonal_final_v1.html")

In [12]:
fig_sigma_tot_eik = go.Figure()

add_iterative_curve(fig_sigma_tot_eik, lst_sqrt_s, lst_sigma_tot_eik, 'Sigma Tot Eikonal vs sqrt(s)', 'sqrt s [GeV]', 'Sigma Tot Eikonal [mb]', 'Sigma Tot Eikonal')


add_iterative_curve(fig_sigma_tot_eik, lst_sqrt_s, lst_sigma_tot_born, 'Sigma Tot vs sqrt(s)', 'sqrt s [GeV]', 'Sigma Tot [mb]', 'Sigma Tot born')




fig_sigma_tot_eik.add_trace(go.Scatter(
        x=x_atlas,
        y=y_atlas,
        mode='markers',
        marker=dict(
            color='black',
            size=6,
            symbol='square'
        ),
        error_y=dict(
            type='data',
            array=y_error_atlas,
            visible=True
        ),
        name='ATLAS Data'
    ))

fig_sigma_tot_eik.show(renderer = 'browser')
# fig_sigma_tot_eik.write_html("sigma_tot_eikonal_final_v1.html")
# fig.write_image(save_path, width=1200, height=600)